## Exercise 1: Tuning, Blending and Deployment
**Dataset Used:** Employee Attrition (PyCaret Built-in)

1. Select Top-3 models. Tune each with `tune_model`.
2. Blend the 3 tuned models with `blend_models()`.
3. Save best model and load it. Predict on new data.

## Dataset Validation
Run this cell to verify that your datasets are present and correctly formatted.

In [ ]:
# --- DATASET VALIDATION ---
import os
import pandas as pd

def validate_dataset(filepath, expected_columns=None, avoid_columns=None):
    if not os.path.exists(filepath):
        print(f'❌ ERROR: Dataset not found at {filepath}')
        return False
    try:
        df = pd.read_csv(filepath, nrows=5)
        print(f'✅ SUCCESS: Dataset found at {filepath} (Columns: {df.shape[1]})')
        if expected_columns:
            missing = [c for c in expected_columns if c not in df.columns]
            if missing:
                print(f'⚠️ WARNING: Missing expected columns: {missing}')
                return False
        if avoid_columns:
            forbidden = [c for c in avoid_columns if c in df.columns]
            if forbidden:
                print(f'❌ ERROR: Found forbidden columns {forbidden}. Wrong dataset!')
                return False
        return True
    except Exception as e:
        print(f'❌ ERROR: Could not read dataset: {e}')
        return False

print('Validation helper loaded. Call validate_dataset(path) before loading your data.')
# validate_dataset('../data/your_dataset.csv')

In [1]:
# # !pip install pycaret
from pycaret.classification import setup, compare_models, tune_model, blend_models, save_model, load_model, predict_model
from pycaret.datasets import get_data

# Load sample dataset
import pandas as pd
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=500, n_features=5, random_state=42)
data = pd.DataFrame(X, columns=['f1', 'f2', 'f3', 'f4', 'f5'])
data['Class variable'] = y
clf = setup(data=data, target='Class variable', session_id=123, verbose=False)

# 1. Top 3 models and Tuning
print("Finding top 3 models...")
top3 = compare_models(exclude=['catboost'], n_select=3, sort='AUC')

print("\nTuning top 3 models...")
# Reduced n_iter for speed in demonstration
tuned_top3 = [tune_model(m, n_iter=10, choose_better=True) for m in top3]

# 2. Blending
print("\nBlending models...")
blender = blend_models(estimator_list=tuned_top3)

# 3. Save, Load, Predict
print("\nSaving model...")
save_model(blender, 'blended_pipeline')

print("Loading model...")
loaded_model = load_model('blended_pipeline')

print("Predicting on new data (using a sample of original for demo)...")
predictions = predict_model(loaded_model, data=data.tail())
print(predictions[['prediction_label', 'prediction_score']])

print("\nREST API Deployment:")
print("To deploy this via REST API, you would use FastAPI or Flask. You load the model using `load_model()`, then expose an endpoint (e.g., POST /predict) that takes JSON data, converts it to a DataFrame, and calls `predict_model()`. PyCaret even has a built-in `create_api()` function for this!")


Finding top 3 models...


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
et,Extra Trees Classifier,0.9343,0.9837,0.9320,0.9388,0.9346,0.8686,0.8701,0.0230
rf,Random Forest Classifier,0.9343,0.9827,0.9261,0.9427,0.9334,0.8686,0.8704,0.0260
gbc,Gradient Boosting Classifier,0.9343,0.9814,0.9320,0.9381,0.9341,0.8686,0.8704,0.0160
lightgbm,Light Gradient Boosting Machine,0.9371,0.9797,0.9376,0.9374,0.9366,0.8743,0.8760,0.0440
xgboost,Extreme Gradient Boosting,0.9543,0.9752,0.9546,0.9564,0.9542,0.9086,0.9109,0.0110
ada,Ada Boost Classifier,0.9400,0.9732,0.9261,0.9550,0.9394,0.8800,0.8821,0.0130
nb,Naive Bayes,0.8686,0.9660,0.8369,0.9056,0.8647,0.7379,0.7476,0.0050
knn,K Neighbors Classifier,0.9029,0.9528,0.8869,0.9241,0.9024,0.8058,0.8111,0.3090
ridge,Ridge Classifier,0.8771,0.9507,0.8814,0.8824,0.8788,0.7546,0.7600,0.0050
lda,Linear Discriminant Analysis,0.8771,0.9507,0.8814,0.8824,0.8788,0.7546,0.7600,0.0040



Tuning top 3 models...


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8857,0.9869,0.8889,0.8889,0.8889,0.7712,0.7712
1,0.9429,0.9902,0.8889,1.0000,0.9412,0.8860,0.8918
2,0.9714,1.0000,0.9444,1.0000,0.9714,0.9429,0.9444
3,0.8857,0.9771,0.8333,0.9375,0.8824,0.7720,0.7771
4,0.9143,0.9837,0.9444,0.8947,0.9189,0.8282,0.8295
5,0.9143,0.9608,0.8889,0.9412,0.9143,0.8287,0.8301
6,0.8857,0.9771,0.8824,0.8824,0.8824,0.7712,0.7712
7,0.9714,1.0000,1.0000,0.9444,0.9714,0.9429,0.9444
8,0.9429,1.0000,1.0000,0.8947,0.9444,0.8860,0.8918


Fitting 10 folds for each of 10 candidates, totalling 100 fits


...

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9429,0.9902,0.9444,0.9444,0.9444,0.8856,0.8856
1,0.9429,0.9902,0.9444,0.9444,0.9444,0.8856,0.8856
2,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
3,0.9429,0.9739,0.8889,1.0000,0.9412,0.8860,0.8918
4,0.9429,0.9902,0.9444,0.9444,0.9444,0.8856,0.8856
5,0.9143,0.9837,0.9444,0.8947,0.9189,0.8282,0.8295
6,0.9429,0.9935,0.8824,1.0000,0.9375,0.8852,0.8911
7,0.9714,1.0000,1.0000,0.9444,0.9714,0.9429,0.9444
8,0.9714,0.9967,1.0000,0.9444,0.9714,0.9429,0.9444



Saving model...
Transformation Pipeline and Model Successfully Saved
Loading model...


Transformation Pipeline and Model Successfully Loaded
Predicting on new data (using a sample of original for demo)...


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000


     prediction_label  prediction_score
495                 0            0.9479
496                 1            1.0000
497                 0            0.9932
498                 0            0.9776
499                 1            1.0000

REST API Deployment:
To deploy this via REST API, you would use FastAPI or Flask. You load the model using `load_model()`, then expose an endpoint (e.g., POST /predict) that takes JSON data, converts it to a DataFrame, and calls `predict_model()`. PyCaret even has a built-in `create_api()` function for this!
